In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv
# Production layout: add project root and src for imports (run from repo root or notebooks/ingestion/)
_root = Path(".").resolve()
if _root.name == "ingestion":
    _root = _root.parent.parent
elif (_root / "src").is_dir():
    pass
else:
    _root = _root.parent
load_dotenv(_root / ".env")
load_dotenv("/app/.env")
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "src"))

from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries
from storage.postgres.news_dataframe import (
    filter_financial_news_by_date,
    filter_financial_news_ingested_today,
    filter_financial_news_published_today,
    get_financial_news_content_by_id,
    normalize_financial_news_datetime_column,
)
from storage.cloud.CloudStorage import CloudStorageProvider

import pandas as pd
from datetime import date, datetime

# Sanity check: if this fails, use File → Reload Notebook from Disk, then restart kernel
import storage.postgres.news_dataframe as _news_df
print(f"Using news_dataframe from: {_news_df.__file__}")
print(f"filter_financial_news_ingested_today: OK")

Using news_dataframe from: /app/src/storage/postgres/news_dataframe.py
filter_financial_news_ingested_today: OK


In [2]:
table_name = PostgresSQL_table_queries.FINANCIAL_NEWS_TABLE_NAME
pg_conn = PgConn(table_name)
df = pg_conn.get_financial_news()
if df is None:
    raise RuntimeError(
        "get_financial_news() failed — check the error printed above "
        "(connection, table name, or schema)."
    )

Connection to the database successful!
Table name set to: financial_news_241118


In [3]:
print(f"Total news articles queried: {df.shape[0]}")

Total news articles queried: 97


In [4]:
df.head()

,id,source,headline,href,summary,content,author,minsread,datetime,created_at
0,2095050064461797250,decrypt,$161 Million in Decade-Old Bitcoin Has Moved i...,https://finance.yahoo.com/markets/crypto/artic...,,The steady procession of long-dormant Bitcoin ...,Decrypt Staff,2 min read,2026-09-22 21:16:03,2026-09-23 03:05:46.315952
1,189562798412200229,BeInCrypto,2 Altcoins Just Got Wall Street's Stamp of App...,https://finance.yahoo.com/markets/crypto/artic...,,Photo by BeInCrypto\nCME Group will add Bitcoi...,Harsh Notariya,2 min read,2026-09-22 12:49:59,2026-09-23 03:05:46.499128
2,723427789064364733,Simply Wall St.,3 Crypto Stocks To Watch In September 2026,https://finance.yahoo.com/markets/crypto/artic...,,AI-fuelled enthusiasm has swung global markets...,Sasha Jovanovic,4 min read,2026-09-22 14:15:19,2026-09-23 03:05:46.440281
3,1094872235512492095,BeInCrypto,3 Meme Coins to Watch in the Fourth Week of Se...,https://finance.yahoo.com/markets/crypto/artic...,,"Photo by BeInCrypto\nDogwifhat (WIF), Pepe (PE...",Harsh Notariya,4 min read,2026-09-22 16:13:51,2026-09-23 03:05:46.406672
4,3911483580991441546,CCN,"69,000 Bitcoin Holders Panic-Sold — Then BTC S...",https://finance.yahoo.com/markets/crypto/artic...,,Bitcoin topped $85K after smaller wallet count...,Giuseppe Ciccomascolo,5 min read,2026-09-22 12:32:19,2026-09-23 03:05:46.504212


In [5]:
def delete_records_for_current_date(df, pg_conn):
    if df is None:
        print("DataFrame is empty. No records to delete.")
        return
        
    today_records = filter_financial_news_by_date(df)

    # Extract date strings (DB/delete API expects stored string form)
    date_strings = today_records['datetime'].astype(str).tolist()

    # Call the delete_records_by_date method
    pg_conn.delete_records_by_date(date_strings)

# Then call the delete_records_for_current_date method
#delete_records_for_current_date(df, pg_conn)
#list_ids = ["11111"]
#pg_conn.delete_records_by_ids(list_ids)

In [6]:
class DataETL():
    
    def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
    class Export():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def set_dataframe(self, dataframe):
            self.df = dataframe
        
        def export_text_to_s3(self, bucket_name, prefix_path, file_format):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()

            # Create a new bucket
            aws_storage.create_bucket(bucket_name)

            # Upload DataFrame with datetime subfolder structure
            aws_storage.upload_dataframe_with_datetime_subfolders(self.df, bucket_name, prefix_path, file_format)
        
        def export_text_to_s3_full_file(self, bucket_name, prefix_path, filename):
            aws_storage = self.cloudProvider.AWS()
            aws_storage.upload_dataframe_to_csv(self.df, bucket_name, filename, prefix_path)
            
    class Ingestion():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def get_full_data_csv_file(self, bucket_name, prefix_path):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_csv_from_specific_folder(bucket_name, prefix_path)
        
        def get_data_csv_file_by_datetime(
            self, bucket_name, prefix_path, year, month, day, hour=None, minute=None, second=None
        ):
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_dataframe_from_specific_datetime(
                bucket_name,
                prefix_path,
                year=year,
                month=month,
                day=day,
                hour=hour,
                minute=minute,
                second=second,
            )
    
    class Process():
        
        def __init__(self, dataframe):
            self.df = dataframe
        
        def getData():
            self.df = pg_conn.get_financial_news()
        
        def filter_by_current_date(self):
            # Article publish date (datetime) on today's calendar date (default on_date=None)
            return filter_financial_news_by_date(self.df)
            
    class Transform():
        def extractStopWords():
            pass

In [7]:
etl = DataETL(df)

# Export by article publish time (datetime). on_date=None → today's local date (date.today()).
export_on_date = "2026-09-22"  # e.g. "2026-05-23" to override today
filtered_df = filter_financial_news_by_date(df, on_date=export_on_date)

print(
    f"Filter date (datetime column): {export_on_date or date.today()} | "
    f"Rows matched: {len(filtered_df)} | "
    f"Ingested today (created_at only): {len(filter_financial_news_ingested_today(df))}"
)
filtered_df.head()

Filter date (datetime column): 2026-09-22 | Rows matched: 95 | Ingested today (created_at only): 97


,id,source,headline,href,summary,content,author,minsread,datetime,created_at
0,2095050064461797250,decrypt,$161 Million in Decade-Old Bitcoin Has Moved i...,https://finance.yahoo.com/markets/crypto/artic...,,The steady procession of long-dormant Bitcoin ...,Decrypt Staff,2 min read,2026-09-22 21:16:03,2026-09-23 03:05:46.315952
1,189562798412200229,BeInCrypto,2 Altcoins Just Got Wall Street's Stamp of App...,https://finance.yahoo.com/markets/crypto/artic...,,Photo by BeInCrypto\nCME Group will add Bitcoi...,Harsh Notariya,2 min read,2026-09-22 12:49:59,2026-09-23 03:05:46.499128
2,723427789064364733,Simply Wall St.,3 Crypto Stocks To Watch In September 2026,https://finance.yahoo.com/markets/crypto/artic...,,AI-fuelled enthusiasm has swung global markets...,Sasha Jovanovic,4 min read,2026-09-22 14:15:19,2026-09-23 03:05:46.440281
3,1094872235512492095,BeInCrypto,3 Meme Coins to Watch in the Fourth Week of Se...,https://finance.yahoo.com/markets/crypto/artic...,,"Photo by BeInCrypto\nDogwifhat (WIF), Pepe (PE...",Harsh Notariya,4 min read,2026-09-22 16:13:51,2026-09-23 03:05:46.406672
4,3911483580991441546,CCN,"69,000 Bitcoin Holders Panic-Sold — Then BTC S...",https://finance.yahoo.com/markets/crypto/artic...,,Bitcoin topped $85K after smaller wallet count...,Giuseppe Ciccomascolo,5 min read,2026-09-22 12:32:19,2026-09-23 03:05:46.504212


In [8]:
print(f"Total filtered news articles queried: {filtered_df.shape[0]}")

Total filtered news articles queried: 95


In [9]:
import os

export_rows_to_s3 = True
etl_export = etl.Export(filtered_df)
bucket_name = "test-financial-news-bucket"
prefix_path = "news/crypto"
file_format = "csv"

if export_rows_to_s3 and not filtered_df.empty:
    if not os.getenv("AWS_ACCESS_KEY_ID") or not os.getenv("AWS_SECRET_ACCESS_KEY"):
        raise RuntimeError(
            "AWS credentials not configured. Uncomment and set AWS_ACCESS_KEY_ID and "
            "AWS_SECRET_ACCESS_KEY in .env, then restart Jupyter: ./docker/start_jupyter.ps1"
        )
    export_df = normalize_financial_news_datetime_column(filtered_df)
    etl_export.set_dataframe(export_df)
    etl_export.export_text_to_s3(bucket_name, prefix_path, file_format)

Bucket 'test-financial-news-bucket' already exists.
Data for row 0 with id '2095050064461797250' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=09/day=22/hour=21/minute=16/second=03/format=csv/2095050064461797250.csv'
Data for row 1 with id '189562798412200229' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=09/day=22/hour=12/minute=49/second=59/format=csv/189562798412200229.csv'
Data for row 2 with id '723427789064364733' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=09/day=22/hour=14/minute=15/second=19/format=csv/723427789064364733.csv'
Data for row 3 with id '1094872235512492095' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=09/day=22/hour=16/minute=13/second=51/format=csv/1094872235512492095.csv'
Data for row 4 with id '3911483580991441546' uploaded to S3 bucket 'test-financial-news-bucket' under fo

In [10]:
post_full_csv = False
if (post_full_csv == True) and not filtered_df.empty:
    now = datetime.now()
    filename = f"{now.year}-{now.month:02}-{now.day:02}_full_record"
    etl_export.set_dataframe(etl.df)
    etl_export.export_text_to_s3_full_file(bucket_name, prefix_path, filename)

In [11]:
ingest_data = False
get_full_file = False
get_by_datetime = True
df_from_file = None

if ingest_data == True:
    etl_ingestion = etl.Ingestion(etl.df)
    bucket_name = "test-financial-news-bucket"
    prefix_path = "news/crypto/"
    year = '2026'
    month = '05'
    day = '24'
    hour = None   # set e.g. '04' to narrow to one hour; None = whole day
    minute = None
    if get_full_file == True:
        filename = f"{year}-{month}-{day}_full_record.csv"
        full_path = f"{prefix_path}{filename}"
        df_from_file = etl_ingestion.get_full_data_csv_file(bucket_name, full_path)
    elif get_by_datetime == True:
        df_from_file = etl_ingestion.get_data_csv_file_by_datetime(bucket_name, prefix_path, year, month, day, hour, minute)

In [12]:
if df_from_file is not None:
    print(df_from_file.count())
    df_from_file.head()

In [13]:
targetId = ""  # i.e. "1221589746717124508" targetId can be string or numeric type

# Lookup order: S3 ingest result, filtered export batch, then full DB pull
lookup_df = None
lookup_source = None
for name, candidate in (
    ("df_from_file", df_from_file if "df_from_file" in dir() else None),
    ("filtered_df", filtered_df if "filtered_df" in dir() else None),
    ("df", df if "df" in dir() else None),
):
    if candidate is not None and not getattr(candidate, "empty", True):
        lookup_df = candidate
        lookup_source = name
        break

if targetId and lookup_df is not None:
    full_content = get_financial_news_content_by_id(lookup_df, targetId)
    if full_content:
        print(full_content)
    else:
        print(
            f"No content for id {targetId!r} in {lookup_source} "
            f"({len(lookup_df)} rows). Id column dtype: {lookup_df['id'].dtype}"
        )
else:
    print("Set targetId and ensure df_from_file, filtered_df, or df is loaded.")

Set targetId and ensure df_from_file, filtered_df, or df is loaded.
